In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 모델 다운로드 → 바로 드라이브에 저장
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="AI-MO/Kimina-Prover-Distill-1.7B",
    local_dir="/content/drive/MyDrive/models/kimina-prover-1.7b"
)
print("완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

완료!


In [3]:
import subprocess
# elan 설치
subprocess.run(
    'curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y',
    shell=True
)
import os
os.environ["PATH"] = f"/root/.elan/bin:{os.environ['PATH']}"

# 확인
result = subprocess.run(["lean", "--version"], capture_output=True, text=True)
print(result.stdout)

Lean (version 4.30.0, x86_64-unknown-linux-gnu, commit d024af099ca4bf2c86f649261ebf59565dc8c622, Release)



In [13]:
def run_lean(code: str, timeout: int = 30) -> dict:
    with open("_tmp.lean", "w") as f:
        f.write(code)
    try:
        result = subprocess.run(
            ["lean", "_tmp.lean"],
            capture_output=True, text=True,
            timeout=timeout
        )
        return {"stdout": result.stdout, "stderr": result.stderr, "returncode": result.returncode}
    except subprocess.TimeoutExpired:
        return {"stdout": "", "stderr": "TIMEOUT", "returncode": -1}

def classify_error(output: dict) -> str:
    if output["stderr"] == "TIMEOUT":
        return "PROVER_FAILURE_timeout"

    msg = output["stdout"] + output["stderr"]

    if "sorry" in msg and "warning" in msg:
        return "SORRY"
    if output["returncode"] == 0:
        return "SUCCESS"

    syntactic = [
        "synthInstanceFailed",
        "failed to synthesize",
        "unexpected token",
        "unknown identifier",
        "application type mismatch",
        "type mismatch",
    ]
    prover = [
        "omega could not prove",
        "unsolved goals",
        "unknown tactic",
        "tactic failed",
        "counterexample",
        "unknown constant",
        "Tactic `rfl` failed",      # ← 새로 추가
        "is not definitionally equal",  # ← 새로 추가
        "failed to prove",          # ← 새로 추가
    ]

    if any(p in msg for p in syntactic):
        return "SYNTACTIC"
    if any(p in msg for p in prover):
        return "PROVER_FAILURE"

    return "UNKNOWN"

def extract_proof(generated: str) -> str:
    # 첫 번째 빈 줄 또는 자연어 시작 전까지만 추출
    lines = generated.split('\n')
    proof_lines = []
    for line in lines:
        # 자연어 시작 감지
        if any(line.startswith(kw) for kw in [
            'The ', 'This ', 'Let', 'In ', 'First', 'Note',
            'We ', 'Now ', 'Here', 'So ', 'Since'
        ]):
            break
        proof_lines.append(line)
    return '\n'.join(proof_lines).strip()

def generate_proof(nl_problem: str, fl_statement: str) -> str:
    prompt = f"Problem: {nl_problem}\nFormal:\n{fl_statement}"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    print(f"[raw 출력]\n{raw}\n")
    proof = extract_proof(raw)
    print(f"[파싱된 proof]\n{proof}\n")
    return proof

In [10]:
prob = {
    "id": "prob_001",
    "nl": "For any natural number n, n + 0 equals n.",
    "fl": "theorem prob_001 : ∀ n : Nat, n + 0 = n := by"
}

proof = generate_proof(prob["nl"], prob["fl"])
print(f"생성된 proof:\n{proof}")

full_code = prob["fl"] + "\n  " + proof
lean_out = run_lean(full_code)
error_type = classify_error(lean_out)

print(f"\n분류 결과: {error_type}")
print(f"Lean 출력: {(lean_out['stdout'] + lean_out['stderr'])[:300]}")

[raw 출력]
intro n
  simp
The problem asks to complete a Lean 4 proof for the theorem `∀ n : Nat, n + 0 = n`.
The theorem states that for any natural number `n`, adding 0 to `n` results in `n`. This is a fundamental property of natural numbers.

Let's think about how to prove this in Lean 4.

First, we need to understand the structure of the proof. The theorem is a universal statement: `∀ n : Nat, n + 0 = n`. In Lean, we can prove this by:
1. Introducing an arbitrary natural number `n`
2. Showing that `n + 0 = n` holds for this arbitrary `n`

In Lean 4, we can use the `intro` tactic to introduce `n` as a variable. Then we need to prove the equality `n + 0 = n`.

Looking at the goal state after `intro n`, we need to prove `n + 0 = n`. This is a simple equality that should be provable by reflexivity or simplification.

In Lean 4, the `simp` tactic is powerful enough to solve goals involving basic arithmetic. It will simplify `n + 0` to `

[파싱된 proof]
intro n
  simp

생성된 proof:
intro n
  si

In [14]:
# Cell 5 — 여러 문제 테스트
import json

test_problems = [
    {
        "id": "prob_001",
        "nl": "For any natural number n, n + 0 equals n.",
        "fl": "theorem prob_001 : ∀ n : Nat, n + 0 = n := by"
    },
    {
        "id": "prob_002",
        "nl": "For any natural number n, n + 1 is greater than n.",
        "fl": "theorem prob_002 : ∀ n : Nat, n + 1 > n := by"
    },
    {
        "id": "prob_003",
        "nl": "1 + 1 equals 2.",
        "fl": "theorem prob_003 : 1 + 1 = 2 := by"
    },
    {
        "id": "prob_004",
        "nl": "For any natural number n, n equals n.",
        "fl": "theorem prob_004 : ∀ n : Nat, n = n := by"
    },
    {
        "id": "prob_005",  # 일부러 틀린 statement
        "nl": "For any natural number n, n + 1 equals n.",
        "fl": "theorem prob_005 : ∀ n : Nat, n + 1 = n := by"
    },
]

results = []
for prob in test_problems:
    print(f"\n--- {prob['id']} ---")
    proof = generate_proof(prob["nl"], prob["fl"])
    full_code = prob["fl"] + "\n  " + proof
    lean_out = run_lean(full_code)
    error_type = classify_error(lean_out)
    print(f"분류 결과: {error_type}")

    results.append({
        "id": prob["id"],
        "nl": prob["nl"],
        "proof": proof,
        "error_type": error_type,
        "lean_output": (lean_out["stdout"] + lean_out["stderr"])[:200]
    })

print("\n===== 최종 결과 =====")
for r in results:
    print(f"{r['id']}: {r['error_type']}")

# 드라이브에 저장
with open("/content/drive/MyDrive/models/results_colab.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\n드라이브에 저장 완료!")


--- prob_001 ---
[raw 출력]
intro n
  simp
The problem asks to complete a Lean 4 proof for the theorem `∀ n : Nat, n + 0 = n`.
The theorem states that for any natural number `n`, adding 0 to `n` results in `n`. This is a fundamental property of natural numbers.

Let's think about how to prove this in Lean 4.

First, we need to understand the structure of the proof. The theorem is a universal statement: `∀ n : Nat, n + 0 = n`. In Lean, we can prove this by:
1. Introducing an arbitrary natural number `n`
2. Showing that `n + 0 = n` holds for this arbitrary `n`

In Lean 4, we can use the `intro` tactic to introduce `n` as a variable. Then we need to prove the equality `n + 0 = n`.

Looking at the goal state after `intro n`, we need to prove `n + 0 = n`. This is a simple equality that should be provable by reflexivity or simplification.

In Lean 4, the `simp` tactic is powerful enough to solve goals involving basic arithmetic. It will simplify `n + 0` to `

[파싱된 proof]
intro n
  simp

분류 결과

In [12]:
# Cell 6 — prob_005 raw 에러 확인
full_code = "theorem prob_005 : ∀ n : Nat, n + 1 = n := by\n  intro n\n  rfl"
lean_out = run_lean(full_code)
print("stdout:", lean_out["stdout"])
print("stderr:", lean_out["stderr"])
print("returncode:", lean_out["returncode"])

stdout: _tmp.lean:3:2: error: Tactic `rfl` failed: The left-hand side
  n + 1
is not definitionally equal to the right-hand side
  n

n : Nat
⊢ n + 1 = n

stderr: 
returncode: 1


In [8]:
def extract_proof(generated: str) -> str:
    # 첫 번째 빈 줄 또는 자연어 시작 전까지만 추출
    lines = generated.split('\n')
    proof_lines = []
    for line in lines:
        # 자연어 시작 감지
        if any(line.startswith(kw) for kw in [
            'The ', 'This ', 'Let', 'In ', 'First', 'Note',
            'We ', 'Now ', 'Here', 'So ', 'Since'
        ]):
            break
        proof_lines.append(line)
    return '\n'.join(proof_lines).strip()

def generate_proof(nl_problem: str, fl_statement: str) -> str:
    prompt = f"Problem: {nl_problem}\nFormal:\n{fl_statement}"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    print(f"[raw 출력]\n{raw}\n")
    proof = extract_proof(raw)
    print(f"[파싱된 proof]\n{proof}\n")
    return proof

In [6]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "/content/drive/MyDrive/models/kimina-prover-1.7b"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("모델 로딩 완료!")
print(f"Device: {next(model.parameters()).device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


모델 로딩 완료!
Device: cuda:0
